# Lab 3 — SCD Type 1 con AUTO CDC INTO

Este notebook define un pipeline SDP que aplica **CDC** sobre un catálogo de productos usando **SCD Type 1** (sin historia).

**Diferencia clave con Lab 2:**  
- Lab 2: `stored_as_scd_type=2` — guarda historial completo con `__START_AT` y `__END_AT`.  
- Lab 3: `stored_as_scd_type=1` — solo conserva el **estado actual**. Cada UPDATE sobreescribe el registro. Cada DELETE elimina la fila.

> Este notebook NO se ejecuta standalone. Debe asociarse a un pipeline SDP en la UI de Databricks.

## SCD Type 1 vs Type 2 — Cuándo usar cada uno

| Criterio | SCD Type 1 | SCD Type 2 |
|---|---|---|
| Historia | No se conserva | Registro completo con fechas |
| Filas por entidad | 1 (estado actual) | N (una por cada cambio) |
| Columnas extra | Ninguna | `__START_AT`, `__END_AT` |
| Caso de uso | Precio actual, estado de stock | Auditoría, snapshots, cumplimiento |
| Complejidad de query | Baja | Media (filtrar `__END_AT IS NULL`) |

**Regla práctica:** si necesitas responder *¿cuál es el valor HOY?*, usa Type 1.  
Si necesitas *¿cuál era el valor el 15 de enero?*, usa Type 2.

**Dataset de este lab:** catálogo de 20 productos con cambios de precio, estado de stock y proveedor durante enero–febrero 2025.

## Configuración del Pipeline en la UI

1. Workspace → **New** → **ETL Pipeline**
2. **Name:** `dbassociate_s08_lab3`
3. **Source code:** este notebook
4. **Catalog:** `dbassociate` — **Target schema:** dejar vacío (el notebook usa full qualified names)
5. **Compute:** Serverless
6. **Mode:** Triggered
7. **Channel:** Current

> El pipeline crea automáticamente las tablas declaradas. No es necesario crear los schemas manualmente.

## Carga del Dataset

Sube `productos_catalogo.csv` al Volume antes de ejecutar el pipeline:

```
/Volumes/dbassociate/default/vol_landing/sesion_08/productos/productos_catalogo.csv
```

El archivo contiene **60 eventos CDC** sobre 20 productos únicos:
- 20 `INSERT` — alta inicial de cada producto
- 38 `UPDATE` — cambios de precio, estado de stock y proveedor (PRD-005 cambia de GlobalTech a ImportPeru; PRD-012 cambia de ImportPeru a GlobalTech)
- 2 `DELETE` — productos discontinuados (PRD-011 Cable HDMI, PRD-016 Switch)

Resultado esperado en Silver tras procesar: **18 filas** (20 − 2 eliminados).

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


@dp.table(
    name="dbassociate.bronze.cambios_productos",
    comment="Eventos CDC del catalogo de productos desde el Volume."
)
def bronze_cambios_productos():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option(
                "cloudFiles.schemaLocation",
                "/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_productos"
            )
            .load("/Volumes/dbassociate/default/vol_landing/sesion_08/productos/")
            .withColumn("_ingested_at", F.current_timestamp())
    )

## `dp.create_auto_cdc_flow` con SCD Type 1

La implementación requiere dos pasos separados:

1. **`dp.create_streaming_table`** — declara la tabla destino vacía. SDP no la crea automáticamente para flujos CDC.
2. **`dp.create_auto_cdc_flow`** — define cómo aplicar los cambios.

| Parámetro | Valor | Función |
|---|---|---|
| `target` | `"dbassociate.silver.dim_productos"` | Tabla destino |
| `source` | `"dbassociate.bronze.cambios_productos"` | Tabla fuente |
| `keys` | `["producto_id"]` | Clave de negocio |
| `sequence_by` | `F.col("event_ts")` | Orden temporal de los eventos |
| `apply_as_deletes` | `F.expr("operation = 'DELETE'")` | Expresión que identifica borrados |
| `except_column_list` | `["evento_id", "operation", "event_ts", "_ingested_at"]` | Columnas que NO se persisten en Silver |
| `stored_as_scd_type` | `1` | Sin historia — sobreescribe la fila actual |

**Cambio mínimo respecto a SCD Type 2:** solo `stored_as_scd_type=1` en lugar de `stored_as_scd_type=2`.  
SDP elimina automáticamente las columnas `__START_AT` y `__END_AT` en modo Type 1.

In [0]:
dp.create_streaming_table(
    name="dbassociate.silver.dim_productos",
    comment="Catalogo de productos — estado actual (SCD Type 1, sin historia)"
)

dp.create_auto_cdc_flow(
    target="dbassociate.silver.dim_productos",
    source="dbassociate.bronze.cambios_productos",
    keys=["producto_id"],
    sequence_by=F.col("event_ts"),
    apply_as_deletes=F.expr("operation = 'DELETE'"),
    except_column_list=["evento_id", "operation", "event_ts", "_ingested_at"],
    stored_as_scd_type=1
)

## Verificación (ejecutar en SQL Editor, fuera del pipeline)

```sql
-- 1. Estado actual del catálogo
SELECT producto_id, nombre, categoria, precio, stock_status, proveedor
FROM dbassociate.silver.dim_productos
ORDER BY producto_id;
-- Esperado: 18 filas (20 INSERT - 2 DELETE)

-- 2. Verificar ausencia de columnas de historia
DESCRIBE TABLE dbassociate.silver.dim_productos;
-- NO deben aparecer __START_AT ni __END_AT

-- 3. Productos por categoría con precio promedio actual
SELECT categoria, COUNT(*) AS productos, ROUND(AVG(precio), 2) AS precio_promedio
FROM dbassociate.silver.dim_productos
GROUP BY categoria
ORDER BY categoria;

-- 4. Comparar volumen: Bronze (todos los eventos) vs Silver (estado actual)
SELECT
    (SELECT COUNT(*) FROM dbassociate.bronze.cambios_productos) AS eventos_bronze,
    (SELECT COUNT(*) FROM dbassociate.silver.dim_productos)    AS productos_activos;
-- Esperado: 60 eventos en Bronze, 18 productos en Silver
```

## Contraste: SCD Type 1 vs Type 2

Si completaste el Lab 2, ejecuta estas queries para comparar:

```sql
-- Lab 2 (Type 2): múltiples filas por cliente con historia
SELECT cliente_id, COUNT(*) AS versiones
FROM dbassociate.silver.dim_clientes
GROUP BY cliente_id
ORDER BY versiones DESC
LIMIT 5;
-- Resultado: clientes con 2-4 versiones históricas

-- Lab 3 (Type 1): siempre 1 fila por producto activo
SELECT producto_id, COUNT(*) AS versiones
FROM dbassociate.silver.dim_productos
GROUP BY producto_id;
-- Resultado: siempre 1 (los eliminados no aparecen)

-- Type 2: historia completa de un cliente
SELECT cliente_id, plan, status, __START_AT, __END_AT
FROM dbassociate.silver.dim_clientes
WHERE cliente_id = 'CLI-001'
ORDER BY __START_AT;

-- Type 1: no existe equivalente, la historia se perdió
-- Solo puedes ver el estado actual del producto
SELECT * FROM dbassociate.silver.dim_productos WHERE producto_id = 'PRD-001';
-- precio = 4199.00, proveedor = TechSupply SAC (último UPDATE del Feb 5)
```

## Antipatrones

- **Usar Type 2 por defecto** — si no necesitas historia, Type 1 es más simple y ocupa menos almacenamiento.
- **Omitir `except_column_list`** — sin este parámetro, columnas como `operation` y `event_ts` se persisten en la dimensión, contaminando el modelo.
- **Omitir `apply_as_deletes`** — los eventos DELETE quedan como filas normales en vez de eliminar el registro.
- **Omitir `sequence_by`** — sin orden temporal, si dos eventos llegan desordenados el resultado es no-determinístico.
- **MERGE INTO manual** — AUTO CDC maneja idempotencia, orden y deduplicación automáticamente; un MERGE manual sin `sequence_by` puede procesar eventos fuera de orden.

In [0]:
# Limpieza — descomentar solo si quieres resetear el lab

# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.cambios_productos")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.dim_productos")

# dbutils.fs.rm(
#     "/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_productos",
#     recurse=True
# )